# 09. 랭크, 영공간, 조건수

로봇의 선형 시스템은 항상 완벽하게 풀리지 않는다. 어떤 방향은 움직일 수 없고, 어떤 자세에서는 작은 오차가 크게 증폭된다.
그걸 판단하는 기본 도구가 **rank, nullspace, condition number** 다.

$$rank(A), \qquad Null(A)=\{x\mid Ax=0\}, \qquad \kappa(A)=\frac{\sigma_{max}}{\sigma_{min}}$$

**로보틱스 연결:**
- $rank(J)$ 부족 → 특이 자세, 제어 불가능한 방향
- $Null(J)$ → redundant robot의 secondary task
- condition number 큼 → 역기구학/캘리브레이션 수치 불안정

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import os
os.makedirs('assets', exist_ok=True)

for font_name in ['Nanum Gothic', 'AppleGothic', 'Malgun Gothic']:
    if any(font.name == font_name for font in fm.fontManager.ttflist):
        plt.rcParams['font.family'] = font_name
        break
plt.rcParams['axes.unicode_minus'] = False

## 1. 랭크와 열공간

$Ax=b$에서 $b$가 $A$의 열공간에 있으면 정확히 풀 수 있고, 아니면 최소제곱 해만 가능하다.
랭크는 행렬이 실제로 만들어낼 수 있는 독립 방향의 개수다.

In [ ]:
A_full = np.array([[1.0, 0.2], [0.1, 1.0]])
A_rank1 = np.array([[1.0, 2.0], [0.5, 1.0]])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
unit = np.array([[1, 0], [0, 1]])
for ax, A, title in [(axes[0], A_full, 'rank 2: 평면을 평면으로 보냄'),
                     (axes[1], A_rank1, 'rank 1: 모든 점이 한 직선으로 눌림')]:
    pts = []
    for x in np.linspace(-1, 1, 21):
        pts.append(A @ np.vstack([np.full(21, x), np.linspace(-1, 1, 21)]))
        pts.append(A @ np.vstack([np.linspace(-1, 1, 21), np.full(21, x)]))
    for P in pts:
        ax.plot(P[0], P[1], color='gray', alpha=0.35, lw=0.8)
    cols = A @ unit
    ax.arrow(0, 0, cols[0,0], cols[1,0], head_width=0.05, color='#E85D24', length_includes_head=True)
    ax.arrow(0, 0, cols[0,1], cols[1,1], head_width=0.05, color='#534AB7', length_includes_head=True)
    ax.set_aspect('equal')
    ax.grid(alpha=0.25)
    ax.set_title(title)
    ax.set_xlim(-2.5, 2.5); ax.set_ylim(-2.5, 2.5)

plt.tight_layout()
plt.savefig('assets/09_rank_column_space.png', dpi=150, bbox_inches='tight')
plt.show()

print('rank(A_full)=', np.linalg.matrix_rank(A_full))
print('rank(A_rank1)=', np.linalg.matrix_rank(A_rank1))

## 2. 영공간 — 움직여도 작업공간 변화가 없는 방향

$Ax=0$을 만족하는 $x$들이 nullspace다. 로봇에서 $J\dot{q}=0$이면 관절은 움직이지만 엔드이펙터는 움직이지 않는다.
이 성질을 이용하면 팔꿈치 자세 조정 같은 secondary task를 수행할 수 있다.

In [ ]:
def nullspace(A, tol=1e-10):
    U, S, Vt = np.linalg.svd(A)
    rank = np.sum(S > tol)
    return Vt[rank:].T

# 3개 관절로 2D 위치를 제어하는 단순 3R planar arm
L = np.array([1.0, 0.8, 0.55])
q = np.deg2rad([35.0, -50.0, 70.0])

def fk_3r(q):
    angles = np.cumsum(q)
    x = np.sum(L * np.cos(angles))
    y = np.sum(L * np.sin(angles))
    return np.array([x, y])

def jacobian_3r(q):
    J = np.zeros((2, 3))
    angles = np.cumsum(q)
    for j in range(3):
        J[0, j] = -np.sum(L[j:] * np.sin(angles[j:]))
        J[1, j] =  np.sum(L[j:] * np.cos(angles[j:]))
    return J

J = jacobian_3r(q)
N = nullspace(J)
print('J=')
print(np.round(J, 4))
print('nullspace basis=')
print(np.round(N, 4))
print('J @ n=', np.round(J @ N[:, 0], 10))

# nullspace 방향으로 관절을 움직이면 EE 위치 변화가 1차적으로 거의 0
n = N[:, 0]
scales = np.linspace(-0.8, 0.8, 70)
pts = np.array([fk_3r(q + s*n) for s in scales])
base = fk_3r(q)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(pts[:,0], pts[:,1], color='#E85D24', lw=2.5, label='nullspace motion path')
ax.scatter(base[0], base[1], color='black', s=70, label='base EE')
ax.set_aspect('equal')
ax.grid(alpha=0.25)
ax.set_title('Nullspace 방향 관절 움직임: EE 위치 변화가 작음')
ax.legend()
plt.savefig('assets/09_nullspace_motion.png', dpi=150, bbox_inches='tight')
plt.show()

print('EE drift range:', np.max(np.linalg.norm(pts - base, axis=1)).round(5))

## 3. 조건수 — 역문제가 얼마나 불안정한가

조건수 $\kappa(A)$가 크면 작은 측정 오차가 해에서 크게 증폭된다.
야코비안 기반 역기구학에서는 $\sigma_{min}$이 0에 가까워질수록 관절 속도가 폭발한다.

In [ ]:
L1, L2 = 1.0, 0.8

def jacobian_2r(t1, t2):
    return np.array([
        [-L1*np.sin(t1)-L2*np.sin(t1+t2), -L2*np.sin(t1+t2)],
        [ L1*np.cos(t1)+L2*np.cos(t1+t2),  L2*np.cos(t1+t2)]
    ])

t2_deg = np.linspace(-179, 179, 500)
conds = []
smin = []
for d in t2_deg:
    S = np.linalg.svd(jacobian_2r(np.deg2rad(30), np.deg2rad(d)), compute_uv=False)
    conds.append(S[0] / max(S[-1], 1e-12))
    smin.append(S[-1])
conds = np.array(conds)
smin = np.array(smin)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].semilogy(t2_deg, conds, color='#E85D24', lw=2.5)
axes[0].axvline(0, color='gray', linestyle='--')
axes[0].set_title('2R arm Jacobian condition number')
axes[0].set_xlabel('theta2 (deg)'); axes[0].set_ylabel('condition number')
axes[0].grid(alpha=0.25)

axes[1].plot(t2_deg, smin, color='#534AB7', lw=2.5)
axes[1].axvline(0, color='gray', linestyle='--')
axes[1].set_title('최소 특이값: 0에 가까우면 특이 자세')
axes[1].set_xlabel('theta2 (deg)'); axes[1].set_ylabel('sigma_min')
axes[1].grid(alpha=0.25)
plt.tight_layout()
plt.savefig('assets/09_condition_number.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Damped Least Squares로 수치 폭발 줄이기

특이 자세 근처에서 일반 의사역행렬은 너무 큰 관절 속도를 만들 수 있다.
댐핑을 넣으면 정확도는 조금 포기하지만 안정성이 좋아진다.

$$J^\# = J^T(JJ^T + \lambda^2I)^{-1}$$

In [ ]:
def damped_pinv(J, lam):
    return J.T @ np.linalg.solve(J @ J.T + lam**2 * np.eye(J.shape[0]), np.eye(J.shape[0]))

J_near = jacobian_2r(np.deg2rad(20), np.deg2rad(3))
dx = np.array([0.02, 0.0])

print('near-singular J=')
print(np.round(J_near, 5))
print('condition number=', np.linalg.cond(J_near).round(2))

for lam in [0.0, 0.03, 0.1, 0.3]:
    if lam == 0.0:
        dq = np.linalg.pinv(J_near) @ dx
        achieved = J_near @ dq
    else:
        dq = damped_pinv(J_near, lam) @ dx
        achieved = J_near @ dq
    print(f'lambda={lam:>4}: |dq|={np.linalg.norm(dq):.4f}, achieved={np.round(achieved, 5)}')

## 요약

| 개념 | 수식 | 로보틱스 활용 |
|------|------|---------------|
| Rank | 독립 출력 방향 수 | 특이 자세 판단 |
| Nullspace | $Ax=0$ | redundancy, secondary task |
| Singular values | $\sigma_i$ | 조작 가능도, 방향별 속도 증폭 |
| Condition number | $\sigma_{max}/\sigma_{min}$ | 역문제 안정성 |
| Damped inverse | $J^T(JJ^T+\lambda^2I)^{-1}$ | 특이 자세 근처 안정화 |